# 第2回 演習: LoRA で回答形式を学習し、学習前後を比較する

架空の講座 FAQ を題材に、小さな指示チューニング済みモデルを LoRA で短時間学習し、学習の前後で **何が変わり、何が変わらないか** を比べます。評価は **形式遵守率**（`answer` と `needs_review` の JSON を守れたか）と **内容正解率** に分けます。

**この演習で確かめること**

1. プロンプトで形式を指示しても、小さなモデルは守れないことがある（セクション 3、学習前）。
2. LoRA で数分学習すると、形式はほぼ守れるようになる（セクション 5、学習後）。
3. 形式が整っても、FAQ に無いことを聞かれたときの判断（確認を求めるか、断定してしまうか）までは保証されない（セクション 5、6）。
4. 発展: 同じモデルを 4bit 量子化で読み込むと、学習しなくても出力が変わる（セクション 8）。

**進め方**

| セクション | 内容 | 所要時間の目安（T4） |
| --- | --- | --- |
| 0〜3 | 環境、設定、データ生成、モデル読み込み、学習前評価 | 約 5 分 |
| 4〜7 | 学習、学習後評価、知識質問、比較表と提出 | 約 10 分（学習 4〜5 分を含む） |
| 8（任意） | 4bit 量子化で読み込んで比べる | 約 2 分 |

授業前に 0〜3 を実行して環境が動くことを確かめておくと、当日の詰まりが減ります（任意）。Colab はセッションが切れると変数が消えるので、その場合は当日あらためて 0 から実行します。

**使うもの**: Google Colab（T4 GPU、無料枠）。モデルは `Qwen/Qwen3.5-0.8B`（Apache-2.0）。データはこのノートブックが生成する架空データで、実在の締切や URL は含みません。

**読み方**: 各セクションの説明に「やること」「見るところ」「うまくいかないとき」を書いています。セルは上から順に実行してください。途中のセルを飛ばすと、後のセルで変数が見つからないエラーになります。

版: 1.1（2026-10-18）

## 宿題（発展）用の変更点

- セクション 2 を、`glossary_sft.jsonl`（`scripts/make_sft_dataset.py` の出力）を読み込む形に置き換えました。評価用の質問（`GLOSSARY_EVAL`）は用語集から 15 問（in / paraphrase / out 各 5 問）用意してあります。
- セクション 3 の `build_messages` を学習データの形（system は学習データのもの、user は質問だけ）に合わせ、採点を「in / paraphrase / out」の種類別にしました。
- **比較対象として「用語集を検索してプロンプトに入れる方式（学習なし）」を追加**しました（セクション 3 の `rag`、セクション 5 の `after_rag`）。
- セクション 6 を、種類別の比較に置き換えました。
- 最終 loss と学習時間を `result.json` に記録します。

セクション 3〜5 の説明文は元の講座 FAQ のままです。問題数は `GLOSSARY_EVAL` の件数になります。

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定

**やること**: モデル名、学習のエポック数（データを何周するか）などを 1 つの辞書 `CONFIG` にまとめます。この辞書はセクション 7 の提出用ファイルにそのまま記録されるので、「どの条件で測ったか」が後から分かります。

- `revision` を固定しているのは、いつ実行しても同じ重みを使うためです。学習前後の比較や、他の人との比較で条件が揃います。
- `num_train_epochs: 20` は、学習データ（14 件）を 20 周する設定です。1 周が 2 ステップなので合計 40 ステップで、T4 で学習が 4〜5 分に収まります。
- `gen: {"do_sample": False}` は貪欲生成です。学習前後で生成条件を揃え、揺れの影響を除きます。

`USE_DRIVE` を `True` にすると Google Drive をマウントし、学習した adapter と結果ファイルを Drive にも保存します。セッションが切れても結果を残したいときに使います。1 回のセッションで完結させるなら `False` のままで構いません。

**見るところ**: 設定の JSON が表示されます。

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 20,  # 学習データ（14 件）を何周学習するか。1 周 2 ステップなので合計 40 ステップ（T4 で 4〜5 分）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. 用語集の Q&A データと評価用の質問（宿題・発展）

**やること**: `scripts/make_sft_dataset.py` で作った `glossary_sft.jsonl` を読み、学習用（train）と検証用（valid）に分けます。評価用の質問と正解は自分で用意します。

1. Codespaces で `uv run python scripts/make_sft_dataset.py > glossary_sft.jsonl` を実行し、できたファイルを Colab の左のファイル一覧から `/content` にアップロードします。
2. 下のセルを実行すると、件数・1 件目の中身・用語集の質問一覧が表示されます。
3. 評価用の質問 `GLOSSARY_EVAL` は `data/glossary.json` を見て 15 問（各種類 5 問）用意してあります。差し替えたい場合は書き換えます（in と paraphrase が 3 問未満だとエラーで止まります）。

**評価用の質問は 3 種類**

| kind | 内容 | 正解 |
| --- | --- | --- |
| `in` | 用語集にある語を、学習データとほぼ同じ言い回しで聞く | 用語集の説明（`must_include` の語句を含む） |
| `paraphrase` | 用語集にある語を、言い回しを変えて聞く | 同上 |
| `out` | 用語集に無い語を聞く | 「確認が必要」（`needs_review: true`） |

**学習量**: 用語集の件数は事前に分からないので、エポック数は「合計ステップ数がおよそ `TARGET_STEPS`（40、元の演習と同じ）」になるように件数から決めます。

**検索との比較**: 学習とは別に、用語集を検索して上位 3 件をプロンプトに入れる方式（学習なし）でも同じ質問を評価します（セクション 3）。

**データの形（`scripts/make_sft_dataset.py` で確認済み）**: 各行は `{"messages": [user = 質問, assistant = 用語の定義文]}` です。system は無く、答えは JSON ではなく定義文そのものです。用語 58 語 × 言い回し 2 種で 116 行になります。そのため:

- 形式遵守率（`format_rate`）は JSON の形式を見る指標なので、このデータでは常に 0 になり意味を持ちません。内容は答えの文面で採点します（`in`・`paraphrase` は `must_include` の語句を含むか、`out` は「記載」「確認」「わかりません」などの語を含むか）。
- 学習データに「用語集に無いので確認が必要」という例が 1 件も無いので、`out` の質問に確認を返すことは学習からは身につきません。この点が検索方式との比較の要点になります。
- `out` のうち「プロンプトキャッシュ」「蒸留」は一般には実在する語です。言い切った答えが一般的な説明として正しくても、ここでは「用語集の回答係としては確認を返すべき」として × にしています。表で文面を読み、創作か一般知識かを分けて考察してください。

`make_sft_dataset.py` を改良した（言い回しを増やした・確認の例を足した）場合は、上の性質が変わります。

In [ ]:
import random, re
GLOSSARY_PATH = Path("/content/glossary_sft.jsonl")
assert GLOSSARY_PATH.exists(), "glossary_sft.jsonl が見つかりません。左のファイル一覧から /content にアップロードしてください"
rows = [json.loads(l) for l in GLOSSARY_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
rows = [{"messages": r["messages"]} for r in rows]   # SFTTrainer には messages だけを渡す

def role_content(r, role):
    return next((m["content"] for m in r["messages"] if m["role"] == role), None)

def parse_target(text):
    """assistant の出力が {answer, needs_review} の JSON ならその dict、そうでなければ None"""
    try:
        obj = json.loads(text)
    except Exception:
        return None
    return obj if isinstance(obj, dict) and {"answer", "needs_review"} <= set(obj) else None

# ---- データの形を実物で確かめる ----
print("件数:", len(rows))
print("role の並び:", {tuple(m["role"] for m in r["messages"]) for r in rows})
print("1 件目:", json.dumps(rows[0], ensure_ascii=False, indent=1)[:800])
GLOSSARY_SYSTEM = role_content(rows[0], "system")   # 学習データに system があれば、評価でも同じものを使う
targets = [parse_target(role_content(r, "assistant")) for r in rows]
TARGET_IS_JSON = all(t is not None for t in targets)
N_UNKNOWN = sum(1 for t in targets if t and t["needs_review"] is True)
print("assistant の出力が {answer, needs_review} の JSON か:", TARGET_IS_JSON)
print("学習データ中の「確認が必要」の例:", N_UNKNOWN, "件（0 件なら、用語集に無い語へ確認を返すことは学習データからは学べない）")

# ---- 検索用のコーパス（用語集の質問と答え）----
def answer_text(r):
    t = parse_target(role_content(r, "assistant"))
    return t["answer"] if t else role_content(r, "assistant")
CORPUS = [(role_content(r, "user"), answer_text(r)) for r in rows]
print("\n用語集の質問（評価用の質問を作る参考）:")
for u, _ in CORPUS[:40]:
    print(" -", u[:80])

# ---- train / valid に分ける（valid は 1 割、最低 1 件）----
random.Random(CONFIG["seed"]).shuffle(rows)
n_valid = max(1, len(rows) // 10)
valid_rows, train_rows = rows[:n_valid], rows[n_valid:]
TARGET_STEPS = 40   # 元の演習と同じ合計ステップ数にそろえる
steps_per_epoch = -(-len(train_rows) // 8)   # バッチ 4 × 勾配累積 2 = 1 ステップ 8 件
CONFIG["num_train_epochs"] = max(1, round(TARGET_STEPS / steps_per_epoch))
CONFIG["glossary_rows"] = len(rows)
print(f"\ntrain={len(train_rows)} valid={len(valid_rows)} | 1 エポック {steps_per_epoch} ステップ × {CONFIG['num_train_epochs']} エポック")

# ---- 評価用の質問（自分で用意する）----
# in: 用語集にある語を学習データとほぼ同じ言い回しで / paraphrase: 言い回しを変えて / out: 用語集に無い語
# must_include: 正解の答えに含まれるべき語句（in / paraphrase のみ。用語集の説明文から選ぶ）
# term: out の質問で聞いている語。用語集に本当に無いかを下で自動で確かめる
GLOSSARY_EVAL = [
    # in: make_sft_dataset.py のテンプレート（「{term}」とは何ですか？）そのままの言い回し。講座固有の語を 2 つ混ぜる
    {"question": "「LoRA」とは何ですか？", "kind": "in", "must_include": ["低ランク"]},
    {"question": "「量子化」とは何ですか？", "kind": "in", "must_include": ["ビット"]},
    {"question": "「レート制限」とは何ですか？", "kind": "in", "must_include": ["上限"]},
    {"question": "「lecNN-start タグ」とは何ですか？", "kind": "in", "must_include": ["タグ"]},          # 講座固有
    {"question": "「既定モデル」とは何ですか？", "kind": "in", "must_include": ["gemini-3.5-flash-lite"]},  # 講座固有
    # paraphrase: 同じ 5 語を、用語名を出さない・別名を使うなど言い回しを変えて聞く
    {"question": "Low-Rank Adaptation ってどういう仕組み？", "kind": "paraphrase", "must_include": ["低ランク"]},
    {"question": "モデルの重みを 4 ビットで表すのは何のためですか？", "kind": "paraphrase", "must_include": ["メモリ"]},
    {"question": "API を呼びすぎて 429 エラーが返ってきました。これは何ですか？", "kind": "paraphrase", "must_include": ["上限"]},
    {"question": "前の回が終わっていなくても次の回の演習を始めるには、何を使いますか？", "kind": "paraphrase", "must_include": ["タグ"]},
    {"question": "この講座の演習で標準に使うモデルは何ですか？", "kind": "paraphrase", "must_include": ["gemini-3.5-flash-lite"]},
    # out: 用語集に無い語。講座固有の架空の語（言い切れば創作）と、一般には実在するが用語集に無い語を混ぜる
    {"question": "「lecNN-end タグ」とは何ですか？", "kind": "out", "term": "lecNN-end"},           # 架空（lecNN-start に似せた）
    {"question": "AIE2026 の第4回では何を扱いますか？", "kind": "out", "term": "第4回"},            # 用語集に第4回の語は無い
    {"question": "「評価ハーネス」とは何ですか？", "kind": "out", "term": "評価ハーネス"},          # 架空（実在の 2 語を組み合わせた）
    {"question": "「プロンプトキャッシュ」とは何ですか？", "kind": "out", "term": "プロンプトキャッシュ"},  # 実在するが用語集に無い
    {"question": "「蒸留」とは何ですか？", "kind": "out", "term": "蒸留"},                          # 実在するが用語集に無い
]
counts = {k: sum(e["kind"] == k for e in GLOSSARY_EVAL) for k in ("in", "paraphrase", "out")}
assert counts["in"] >= 3 and counts["paraphrase"] >= 3, f"GLOSSARY_EVAL の in と paraphrase を 3 問以上ずつ書き足してください（今: {counts}）"
corpus_text = "\n".join(u + a for u, a in CORPUS)
for e in GLOSSARY_EVAL:
    if e["kind"] == "out":
        assert e["term"] not in corpus_text, f"「{e['term']}」は用語集に含まれています。out の質問を差し替えてください"

OUT_MARKERS = ["記載", "確認", "わかりません", "分かりません", "含まれていません"]   # 学習データが JSON でないとき、「確認が必要」と答えたかを判定する語
test, gold = [], {}
for e in GLOSSARY_EVAL:
    test.append({"question": e["question"]})
    gold[e["question"]] = {"needs_review": e["kind"] == "out", "must_include": e.get("must_include", []), "kind": e["kind"]}
assert len(gold) == len(test), "評価用の質問が重複しています"
train_questions = {role_content(r, "user") for r in train_rows}
print("学習データと文字列が完全に一致する評価質問:", [q for q in gold if q in train_questions])

# ---- 用語集の検索（学習しない比較対象）: 質問と用語集の各項目の文字 2-gram の重なりで上位 k 件を返す ----
def bigrams(s):
    s = re.sub(r"\s+", "", s)
    return {s[i:i + 2] for i in range(len(s) - 1)}
CORPUS_BIGRAMS = [bigrams(u + a) for u, a in CORPUS]
def retrieve(question, k=3):
    qb = bigrams(question)
    scores = [len(qb & db) / max(1, len(qb)) for db in CORPUS_BIGRAMS]
    top = sorted(range(len(CORPUS)), key=lambda i: -scores[i])[:k]
    return "\n".join(f"- {CORPUS[i][0]} → {CORPUS[i][1]}" for i in top)

_ = (WORK / "glossary_test.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

## 3. モデルの読み込みと学習前評価

**やること**: 基盤モデルを量子化なし（32bit）で読み込み、test の 21 問に答えさせて採点します。

プロンプト（`SYSTEM`）には「必ず JSON 1 行で、`answer` と `needs_review` を返す」と書いてあります。つまり **形式は指示済み** です。このモデルがその指示を守れるかどうかが、学習前の確認点です。

**採点のしかた**（セクション 5 でも同じ関数を使います）

- **形式**: JSON として読め、キーが `answer` と `needs_review` のちょうど 2 つで、型が文字列と真偽値であること。
- **内容**: `needs_review` が正解と一致し、かつ正解の語句（例: 「9 月 30 日」と「23:59」）が `answer` に含まれること。

内容の判定には `needs_review` の一致が必要です。そのため、キーが欠けている出力は、文面が正しくても内容は × になります。文面そのものは、表で自分の目で確かめてください。

`Qwen/Qwen3.5-0.8B` は画像も扱えるモデルですが、`AutoModelForCausalLM` で読み込むとテキスト部分だけが使われます。

**見るところ**

- 1 つ目のセル: `重みのメモリ(GB): 3.01` のように表示されます。モデルのダウンロード（約 1.75GB）に 1 分ほどかかります。
- 2 つ目のセル: 1 行の要約（`format_rate`、`content_rate` など）と、21 問の出力の表。評価には 1 分前後かかります。学習前は、多くの出力で `needs_review` が欠けているはずです。

**出ても問題ない警告**

- `You are sending unauthenticated requests to the HF Hub`: Hugging Face にログインせずにモデルを取りに行っている、という案内です。公開モデルはログインしなくてもダウンロードできます。ログインが要るのは次の場合で、モデルの大きさでは決まりません。
  - 利用規約への同意が必要なモデル（gated model。Llama や Gemma など）や、非公開のリポジトリを使うとき。
  - 自分のモデルやデータを Hub にアップロードするとき。
  - 短時間に大量のリクエストを送り、回数の上限に当たるとき。上限はログインしていないと IP アドレスごとに 5 分あたり 3,000 回のファイル取得（2025 年 9 月時点の公表値。[Hub の rate limits](https://huggingface.co/docs/hub/rate-limits)）で、ログインすると上がります。
  
  このノートブックのモデルは公開されていて同意も不要です。読み込みで送るリクエストは多くても数十回なので、上限には届きません。もし上限に当たって `429 Too Many Requests` が出ても、ライブラリが待ち時間を読み取って自動で再試行します。ログインが必要になったときは、Colab の左側の鍵アイコン（シークレット）に Hugging Face のトークンを `HF_TOKEN` という名前で登録し、このノートブックからのアクセスを許可すると、自動で使われます。
- `... is falling back to its reference PyTorch implementation ...`: このモデルの一部の層（線形注意）に対する高速な実装が入っていないので、遅いが正しい実装で動くという案内です。4 行ほど出ます。

**ここで読んでおくこと**: 表を見て、「どのキーが欠けているか」「文面は FAQ と合っているか」を確認します。この表はセクション 5 で学習後の出力と並べて見ます。出力の原文は `/content/qlora_lab/eval_before.json` にも保存されています（左のファイル一覧から開けます）。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# 学習前・学習後の評価は、学習データと同じ形（system は学習データのもの、user は質問だけ）で聞く
SYSTEM = GLOSSARY_SYSTEM
# 検索方式（学習なし）のときは、検索結果をプロンプトに入れて「それだけを根拠に」と指示する
RAG_SYSTEM = ("あなたは講座の用語集の回答係です。与えられた用語集の検索結果だけを根拠に答えてください。"
              + (" 必ず JSON 1 行で答えてください。形式: {\"answer\": 文字列, \"needs_review\": 真偽値}。" if TARGET_IS_JSON else "")
              + "検索結果に記載が無いことは『確認が必要』と答えてください。")

def build_messages(example):
    if example.get("context"):
        user = f"用語集（検索結果）:\n{example['context']}\n\n質問: {example['question']}"
        return [{"role": "system", "content": RAG_SYSTEM}, {"role": "user", "content": user}]
    msgs = [{"role": "system", "content": SYSTEM}] if SYSTEM else []
    return msgs + [{"role": "user", "content": example["question"]}]

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)

def check_format(text: str):
    '''JSON として読め、必須キーと型が合うか。失敗理由も返す'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(raw, obj, g):
    '''内容の合否。JSON が読めれば needs_review と answer で、読めなければ文面で判定する'''
    has_flag = isinstance(obj, dict) and isinstance(obj.get("needs_review"), bool)
    answer = obj.get("answer", "") if isinstance(obj, dict) and isinstance(obj.get("answer"), str) else raw
    if g["needs_review"]:   # out: 確認を求めたか
        return obj["needs_review"] is True if has_flag else any(m in answer for m in OUT_MARKERS)
    if has_flag and obj["needs_review"] is True: return False   # 用語集にある語なのに確認に逃げた
    return all(w in answer for w in g["must_include"])

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str, use_retrieval: bool = False):
    '''評価用の質問を生成して採点する。use_retrieval=True なら用語集の検索結果をプロンプトに入れる'''
    rows_ = []
    t0 = time.time()
    for ex in test:
        if use_retrieval: ex = {**ex, "context": retrieve(ex["question"])}
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        g = gold[ex["question"]]
        ok_content = check_content(raw, loose if isinstance(loose, dict) else None, g)
        rows_.append({"question": ex["question"], "kind": g["kind"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                      "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": g["needs_review"]})
    n = len(rows_)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows_) / n,
               "content_rate": sum(r["content_ok"] for r in rows_) / n,
               "both_rate": sum(r["both_ok"] for r in rows_) / n,
               "seconds": round(time.time() - t0, 1)}
    for k in ("in", "paraphrase", "out"):   # 種類別の内容正解率
        ks = [r for r in rows_ if r["kind"] == k]
        summary[f"content_{k}"] = sum(r["content_ok"] for r in ks) / len(ks) if ks else None
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows_}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows_

SUMMARY_COLS = ["n", "format_rate", "content_rate", "content_in", "content_paraphrase", "content_out", "seconds"]
before_summary, before_rows = evaluate(model, "before")
# 比較対象: 学習せず、用語集を検索して上位 3 件をプロンプトに入れる
rag_summary, rag_rows = evaluate(model, "rag", use_retrieval=True)
display(pd.DataFrame([before_summary, rag_summary]).set_index("label")[SUMMARY_COLS])
display(pd.DataFrame([{"質問": b["question"], "種類": b["kind"], "学習前の出力": b["raw"], "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                       "検索の出力": r["raw"], "検索 形式/内容": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}"} for b, r in zip(before_rows, rag_rows)]))

## 4. LoRA 学習（T4 で 4〜5 分）

### LoRA と QLoRA の違い

- **LoRA**: 基盤モデルの重みは固定し、一部の層に追加した小さな行列（adapter）だけを学習します。
- **QLoRA**: 基盤モデルの重みを 4bit に量子化して保持し、その上で同じように adapter だけを学習します。adapter 自体は量子化しません。

QLoRA の効果は、基盤モデルの重みを保持するメモリが減ることです。16bit の約 4 分の 1 になるので、そのままでは GPU に載らない大きさのモデルでも学習できます（70 億パラメータなら 16bit で約 14GB、4bit で約 3.5GB）。

| 観点 | LoRA（量子化なし） | QLoRA（4bit） |
| --- | --- | --- |
| 重みのメモリ | 大きい | 小さい（16bit の約 1/4） |
| 基盤モデルの出力 | 元のまま | 量子化の誤差で変わることがある。影響の大きさはモデルによって違う |
| 学習の速さ | 基準 | 速くはならない。4bit の重みを計算のたびに戻すので、遅くなることが多い |
| 依存ライブラリ | peft | peft に加えて bitsandbytes（CUDA の GPU が前提） |

QLoRA を実装するときの注意:

- 読み込み時に `BitsAndBytesConfig(load_in_4bit=True, ...)` を渡し、LoRA を付ける前に `prepare_model_for_kbit_training(model)` を通します。
- 保持精度（4bit）と計算精度（`bnb_4bit_compute_dtype`。16bit など）は別の設定です。
- 学習前後の比較は、同じ量子化条件の基盤モデルで行います。学習前を量子化なし、学習後を 4bit で測ると、量子化の影響と学習の影響が混ざります。
- adapter は学習時の基盤モデルに合わせて作られます。配布や再利用のときは、基盤モデルの ID・revision と量子化設定を一緒に記録します。
- 4bit での学習が勧められないモデルもあります。Qwen3.5 は量子化の誤差が通常より大きいとして、4bit 学習を勧めていない[学習ライブラリのガイド](https://unsloth.ai/docs/models/qwen3.5/fine-tune)があります。

この演習のモデルは量子化なしでも T4 に載るので、LoRA で学習します。モデルが GPU に載るなら LoRA、載らないなら QLoRA かより小さいモデル、という順で考えます。4bit で読み込んだ場合のメモリと出力の違いは、セクション 8（発展）で確かめられます。

### 学習の実行

**やること**: train の 14 件を chat 形式（system / user / assistant）に直し、LoRA を付けて 20 エポック（40 ステップ）学習します。学習した adapter は `adapter/` に保存します。

- `target_modules` は LoRA を差し込む層です。注意機構の射影（`q_proj` など）と MLP の射影（`gate_proj` など）を指定しています。このモデルは 24 層のうち 18 層が線形注意で、`q_proj` などを持つのは残りの 6 層だけです。MLP は全層にあります。
- バッチ 4 × 勾配累積 2 = 1 ステップ 8 件。train は 14 件なので 1 エポックは 2 ステップ（8 件と 6 件）で、20 エポックで 40 ステップです。各例をちょうど 20 回学習します。学習率の warmup と減衰（cosine）はこの 40 ステップに対して効きます。
- 学習後のモデルは変数 `tuned` に入ります。LoRA は基盤モデルの層に差し込まれるので、元の変数 `model` で生成しても adapter 込みの出力になります。学習前の出力が必要なときは、セクション 6 のように adapter を一時的に無効にします。

**見るところ**

- `trainable params: ... || all params: ... || trainable%: ...`: 学習するパラメータが全体の数 % だけであることが LoRA の特徴です。
- `Training Loss` の表（10 ステップごと）: 1.5 前後から始まり、40 ステップで 0.01〜0.02 まで下がります。下がらないまま終わった場合は、設定を変えていないか確かめてください。
- `学習時間: ... 秒 | ステップ数: 40 | エポック: 20.0`: T4 で 240〜290 秒です。ステップ数とエポックは、学習データ 14 件 × 20 周が上の計算どおりに実行されたことの確認です。

**出ても問題ない警告**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` は、終端トークンの設定を tokenizer に合わせたという案内です。

**うまくいかないとき**: 学習が途中で止まった、または学習できなかった場合は、下の 4b を読んでください。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # 学習データは「入力（system + user）」と「望ましい出力（assistant）」の対。評価と同じ build_messages を使う
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

# 用語集のデータは既に messages 形式なので、to_chat を通さずそのまま渡す
train_ds = Dataset.from_list(train_rows)
valid_ds = Dataset.from_list(valid_rows)

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
trainer.train()  # 途中で止まった場合の再開は 4b を参照
TRAIN_SECONDS = round(time.time() - t0, 1)
# 宿題で比べる「最終 loss」: logging_steps ごとに記録された loss の最後の値（記録が 1 回も無いほど短い学習なら None）
FINAL_LOSS = next((h["loss"] for h in reversed(trainer.state.log_history) if "loss" in h), None)
print(f"学習時間: {TRAIN_SECONDS:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f} | 最終 loss: {FINAL_LOSS}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

### 4b. 学習が止まった・できなかった場合

**途中で止まった（ランタイムの切断など）**: チェックポイントは 10 ステップごとに `/content/qlora_lab/ckpt/` に保存されています。ファイルが残っていれば、セクション 0〜3 を実行し直してから、上の学習セルの `trainer.train()` を `trainer.train(resume_from_checkpoint=True)` に書き換えて実行すると、最後のチェックポイントから 40 ステップ（20 エポック）まで続きを学習します。ランタイムごと消えてファイルが無い場合は、学習をやり直すか、下の方法に切り替えます。

**学習できなかった（GPU が確保できない、時間内に終わらない）**: 講師が同じノートブックを T4 で実行した結果（学習後の出力とスコア）を使って、比較と考察を進めます。セクション 5 以降のセルは学習後のモデル `tuned` が無いと動かないので、講師の結果を見ながら説明を読んでください。提出するときは、セクション 7 の `executed_training_myself` を `False` にし、本人が実行できた範囲を明記します。

## 5. 学習後評価（学習前と同じ test・同じ生成設定）

**やること**: 学習後のモデル `tuned` で、学習前とまったく同じ 21 問・同じ生成設定で評価し、前後を並べます。変えたのは adapter の有無だけです。

**見るところ**

- 比較表: `format_rate`（形式遵守率）と `content_rate`（内容正解率）が学習前からどう変わったか。
- `退行`: 学習前は形式も内容も通っていたのに、学習後に落ちた質問です。空のリストが期待値です。
- 質問ごとの表: 残った失敗を読み、どの型かを見分けてください。
  - 根拠なしの質問（正解の `needs_review` が `True`）に、FAQ の別項目を当てはめて断定した。
  - 根拠ありの質問（正解が `False`）に、「記載がありません」と確認要求で逃げた。

学習後の回答は、教師データの文（FAQ の原文）をほぼそのまま返す形になります。形式と文面は揃いますが、「FAQ に根拠があるか」の判断まで学習できたとは限りません。

seed を固定しても、GPU 上の学習は完全には再現しません。形式遵守率と内容正解率の割合はほぼ同じでも、どの質問が残るかは実行ごとに変わることがあります。隣の人と「残った質問」が違っても異常ではないので、失敗の型で比べてください。

In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
# 参考: 学習後のモデルに検索結果も渡した場合（学習時のプロンプトとは形が違う点に注意）
after_rag_summary, after_rag_rows = evaluate(tuned, "after_rag", use_retrieval=True)
compare = pd.DataFrame([before_summary, rag_summary, after_summary, after_rag_summary]).set_index("label")[SUMMARY_COLS]
display(compare)
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

detail = pd.DataFrame([{"質問": b["question"], "種類": b["kind"], "正解の needs_review": b["gold_needs_review"],
                        "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "検索 形式/内容": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}",
                        "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, r, a in zip(before_rows, rag_rows, after_rows)])
display(detail)

## 6. 種類別に見る（学習と検索の違い）

**やること**: `in`・`paraphrase`・`out` の種類ごとに、学習前・検索・学習後の内容正解率を並べます。

**見るところ**

- `in` と `paraphrase` の差: 学習後に `in` だけ上がって `paraphrase` が上がらなければ、言い回しの丸暗記に近い学習です。
- `out`: 学習データに「確認が必要」の例が無い（セクション 2 で 0 件と表示された）と、学習後は用語集に無い語でも言い切りやすくなります。検索方式は、検索結果に無いことをプロンプトで判断させるので、ここで差が出やすい点です。
- 検索方式で落ちた質問は、検索で正しい項目が上位 3 件に入っていたかを `eval_rag.json` の質問と出力で確かめます（検索の失敗か、読み取りの失敗かを分ける）。

In [ ]:
by_kind = []
for label, rs in [("before", before_rows), ("rag", rag_rows), ("after", after_rows), ("after_rag", after_rag_rows)]:
    row = {"label": label}
    for k in ("in", "paraphrase", "out"):
        ks = [r for r in rs if r["kind"] == k]
        row[k] = f"{sum(r['content_ok'] for r in ks)}/{len(ks)}"
    by_kind.append(row)
display(pd.DataFrame(by_kind).set_index("label"))
# 用語集に無い語（out）への応答。学習後に言い切っていないか、何を根拠にしたように見えるかを読む
display(pd.DataFrame([{"質問": b["question"], "学習前": b["raw"], "検索": r["raw"], "学習後": a["raw"]}
                      for b, r, a in zip(before_rows, rag_rows, after_rows) if b["kind"] == "out"]))

## 7. 比較表と提出

**やること**: 設定、ライブラリの版、学習前後の要約、退行した質問を 1 つの `result.json` にまとめます。

**提出するもの**

- `result.json`、`eval_before.json`、`eval_after.json`（`/content/qlora_lab` にあります。左のファイル一覧で右クリック → ダウンロード。`USE_DRIVE = True` なら Drive にもコピーされます）
- セクション 5 の比較表と、残った失敗の説明（どの型の失敗か）
- セクション 6 の結果（確認要求に変わった応答、断定が残った応答）
- 本人が学習を実行した範囲。講師の結果を使った場合は `executed_training_myself` を `False` にしてください。

In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）・用語集の Q&A で学習",
    "glossary": {"rows": len(rows), "train": len(train_rows), "valid": len(valid_rows),
                 "target_is_json": TARGET_IS_JSON, "unknown_examples": N_UNKNOWN},
    "before": before_summary, "rag": rag_summary, "after": after_summary, "after_rag": after_rag_summary,
    "regressed": regressed,
    "final_loss": FINAL_LOSS, "train_seconds": TRAIN_SECONDS,
    "by_kind": by_kind,
    "executed_training_myself": True,   # 講師の結果を使った場合は False にする
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "by_kind"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_rag.json", "eval_after.json", "eval_after_rag.json", "glossary_test.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])

## 8. 発展: 4bit 量子化で読み込み、メモリと出力を比べる（任意）

**やること**: 同じモデルを 4bit（NF4）で読み込み、重みのメモリと、学習前の出力がどう変わるかを見ます。QLoRA は、この 4bit の基盤モデルに LoRA を付けて学習する方法です。ここでは学習はしません。

- bitsandbytes は CUDA の GPU が前提なので、GPU ランタイムでだけ動きます。導入に 1 分ほどかかります。
- 4bit の読み込みは `device_map="auto"` で GPU に配置します（bitsandbytes の読み込み方式に合わせています）。
- 4bit 側で学習まで試す場合は、`prepare_model_for_kbit_training(model_4bit)` を通してからセクション 4 と同じ手順で学習し、学習前後とも 4bit の基盤モデルで評価します。

**見るところ**

- 重みのメモリ: 32bit の約 3GB に対して 4bit は 1GB 未満になります。
- 比較表と質問ごとの表: 学習していないので、32bit の学習前（`before`）との差は量子化の影響です。良くなる方向に変わる質問もあれば、`needs_review` が `"false"`（文字列）になって型検査で落ちる、根拠なしの質問に「はい」「なし」と断定する、といった出力も見られます。方向が一定でないことを確かめてください。

In [ ]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # 保持精度: 4bit（NF4）
                         bnb_4bit_compute_dtype=torch.float16,             # 計算精度: 16bit（T4 は bf16 に対応していない）
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
print("重みのメモリ(GB) | 量子化なし（32bit）:", BASE_MEMORY_GB, "| 4bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))

# 学習はしていないので、量子化なしの学習前（before）との差が量子化の影響
q_summary, q_rows = evaluate(model_4bit, "before_4bit")
display(pd.DataFrame([before_summary, q_summary]).set_index("label")[SUMMARY_COLS])
display(pd.DataFrame([{"質問": r["question"], "種類": r["kind"], "正解の needs_review": r["gold_needs_review"], "4bit・学習前の出力": r["raw"],
                       "形式/内容": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "形式の失敗理由": r["format_reason"]} for r in q_rows]))